# Part 1

### Configuration & Catalog Setup

In [0]:
dbutils.widgets.text("catalog", "de_assessment_dev")
CATALOG = dbutils.widgets.get("catalog")

STORAGE_ACCOUNT = "deassessmentd06fabcc"
RAW_PATH = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.sql(f"USE CATALOG {CATALOG}")

DataFrame[]

### Fetch Shows

In [0]:
import time, requests

def fetch_with_retry(url, max_retries=4, timeout=30):
    """GET with exponential backoff — handles TVMaze 429 rate limits and timeouts."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=timeout)
            if resp.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s before retry {attempt+1}/{max_retries}")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp
        except requests.exceptions.Timeout:
            wait = 2 ** attempt
            print(f"Timeout on {url}. Waiting {wait}s before retry {attempt+1}/{max_retries}")
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise Exception(f"All {max_retries} attempts failed for {url}: {e}")
            time.sleep(2 ** attempt)
    raise Exception(f"Exhausted {max_retries} retries for {url}")

In [0]:
import json
import pandas as pd

response = fetch_with_retry("https://api.tvmaze.com/shows")
shows = response.json()

shows_pdf = pd.DataFrame(shows)
for col_name in shows_pdf.columns:
    if shows_pdf[col_name].dtype == object:
        shows_pdf[col_name] = shows_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

shows_df = spark.createDataFrame(shows_pdf)
print(f"Fetched {len(shows)} shows")

Fetched 240 shows


### Write shows to ADLS Gen 2 Storage

In [0]:
# Write json to raw container using the external location name (Unity Catalog routes auth correctly)
shows_df.write.mode("overwrite").json(f"{RAW_PATH}/shows/")

### Save as shows Bronze Delta Tables

In [0]:
# Read back and save as Bronze Delta table
bronze_shows = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/shows/")

bronze_shows.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.bronze_shows")

print("bronze_shows created:", bronze_shows.count(), "rows")

bronze_shows created: 240 rows


### Fetch Episodes & Cast

In [0]:
import time

show_ids = [show["id"] for show in shows]

# Fetch episodes for all shows
episodes = []
for i, show_id in enumerate(show_ids):
    try:
        response = fetch_with_retry(f"https://api.tvmaze.com/shows/{show_id}/episodes")
        for ep in response.json():
            ep["show_id"] = show_id
            episodes.append(ep)
    except Exception as e:
        print(f"Warning: failed to fetch episodes for show {show_id}: {e}")
    if (i + 1) % 20 == 0:
        print(f"Fetched episodes for {i + 1}/{len(show_ids)} shows")
        time.sleep(1)  # Rate limit courtesy pause

# Fetch cast for all shows
cast = []
for i, show_id in enumerate(show_ids):
    try:
        response = fetch_with_retry(f"https://api.tvmaze.com/shows/{show_id}/cast")
        for member in response.json():
            member["show_id"] = show_id
            cast.append(member)
    except Exception as e:
        print(f"Warning: failed to fetch cast for show {show_id}: {e}")
    if (i + 1) % 20 == 0:
        print(f"Fetched cast for {i + 1}/{len(show_ids)} shows")
        time.sleep(1)  # Rate limit courtesy pause

print(f"Episodes: {len(episodes)}, Cast: {len(cast)}")

Fetched episodes for 20/240 shows
Fetched episodes for 40/240 shows
Fetched episodes for 60/240 shows
Fetched episodes for 80/240 shows
Fetched episodes for 100/240 shows
Fetched episodes for 120/240 shows
Fetched episodes for 140/240 shows
Fetched episodes for 160/240 shows
Fetched episodes for 180/240 shows
Fetched episodes for 200/240 shows
Fetched episodes for 220/240 shows
Fetched episodes for 240/240 shows
Fetched cast for 20/240 shows
Fetched cast for 40/240 shows
Fetched cast for 60/240 shows
Fetched cast for 80/240 shows
Fetched cast for 100/240 shows
Fetched cast for 120/240 shows
Fetched cast for 140/240 shows
Fetched cast for 160/240 shows
Fetched cast for 180/240 shows
Fetched cast for 200/240 shows
Fetched cast for 220/240 shows
Fetched cast for 240/240 shows
Episodes: 30645, Cast: 5100


### Write Episodes & Cast to ADLS

In [0]:
# Flatten complex fields for Spark
episodes_pdf = pd.DataFrame(episodes)
for col_name in episodes_pdf.columns:
    if episodes_pdf[col_name].dtype == object:
        episodes_pdf[col_name] = episodes_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

cast_pdf = pd.DataFrame(cast)
for col_name in cast_pdf.columns:
    if cast_pdf[col_name].dtype == object:
        cast_pdf[col_name] = cast_pdf[col_name].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)

episodes_df = spark.createDataFrame(episodes_pdf)
cast_df = spark.createDataFrame(cast_pdf)

episodes_df.write.mode("overwrite").json(f"{RAW_PATH}/episodes/")
cast_df.write.mode("overwrite").json(f"{RAW_PATH}/cast/")

print(f"Written {episodes_df.count()} episodes and {cast_df.count()} cast members to raw")

Written 30645 episodes and 5100 cast members to raw


### Save Episodes & Cast as Bronze Delta Table

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Explicit schema prevents inferSchema type conflicts across partitions
episode_schema = StructType([
    StructField("_links", StringType()),
    StructField("airdate", StringType()),
    StructField("airstamp", StringType()),
    StructField("airtime", StringType()),
    StructField("id", LongType()),
    StructField("image", StringType()),
    StructField("name", StringType()),
    StructField("number", LongType()),
    StructField("rating", StringType()),
    StructField("runtime", DoubleType()),
    StructField("season", LongType()),
    StructField("show_id", LongType()),
    StructField("summary", StringType()),
    StructField("type", StringType()),
    StructField("url", StringType()),
])

bronze_episodes = spark.read.schema(episode_schema).json(f"{RAW_PATH}/episodes/")
bronze_cast = spark.read.option("inferSchema", "true").json(f"{RAW_PATH}/cast/")

bronze_episodes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.bronze_episodes")

bronze_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.bronze_cast")

print("bronze_episodes:", bronze_episodes.count(), "rows")
print("bronze_cast:", bronze_cast.count(), "rows")

bronze_episodes: 30645 rows
bronze_cast: 5100 rows


In [0]:
# NOT NULL constraints on primary/foreign keys
for tbl, cols in [
    ("bronze_shows",    ["id"]),
    ("bronze_episodes", ["id", "show_id"]),
    ("bronze_cast",     ["show_id"]),
]:
    for col in cols:
        try:
            spark.sql(f"ALTER TABLE {CATALOG}.bronze.{tbl} ALTER COLUMN {col} SET NOT NULL")
            print(f"OK   {tbl}.{col} SET NOT NULL")
        except Exception as e:
            if "already" in str(e).lower():
                print(f"SKIP {tbl}.{col} — already NOT NULL")
            else:
                print(f"WARN {tbl}.{col} — {e}")

# CHECK constraints on identifiers
constraints = [
    ("bronze_shows",    "valid_show_id",   "id > 0"),
    ("bronze_episodes", "valid_episode_id", "id > 0"),
    ("bronze_episodes", "valid_show_ref",   "show_id > 0"),
    ("bronze_cast",     "valid_show_ref",   "show_id > 0"),
]

for tbl, name, expr in constraints:
    try:
        spark.sql(f"ALTER TABLE {CATALOG}.bronze.{tbl} ADD CONSTRAINT {name} CHECK ({expr})")
        print(f"OK   {tbl}: added CHECK {name} ({expr})")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"SKIP {tbl}: {name} already exists")
        else:
            print(f"WARN {tbl}: {name} — {e}")

print("\nBronze schema enforcement applied.")

OK   bronze_shows.id SET NOT NULL
OK   bronze_episodes.id SET NOT NULL
OK   bronze_episodes.show_id SET NOT NULL
OK   bronze_cast.show_id SET NOT NULL
SKIP bronze_shows: valid_show_id already exists
SKIP bronze_episodes: valid_episode_id already exists
SKIP bronze_episodes: valid_show_ref already exists
SKIP bronze_cast: valid_show_ref already exists

Bronze schema enforcement applied.


### Grant access to the Compliance Team

In [0]:
tables = ["bronze_shows", "bronze_episodes", "bronze_cast"]

try:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `Compliance-Team`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.bronze TO `Compliance-Team`")
    for table in tables:
        spark.sql(f"GRANT SELECT ON TABLE {CATALOG}.bronze.{table} TO `Compliance-Team`")
    print("Compliance-Team grants applied successfully")
except Exception as e:
    print(f"Warning: could not apply Compliance-Team grants: {e}")

Compliance-Team grants applied successfully


### Schema evolution & enforcement approach

**Schema enforcement (read):** Episodes are read with an explicit `StructType` schema rather than `inferSchema=True`. This prevents type conflicts when the API returns mixed types across shows (e.g. `runtime` as integer for some shows and null/double for others). By defining types upfront, the read is deterministic and reproducible.

**Schema evolution (write):** Delta writes use `.option("overwriteSchema", "true")` with `.mode("overwrite")`, so if the API adds new fields in the future, the table schema is updated to match the incoming data. This ensures the schema remains flexible and up-to-date, while avoiding duplicate records that would result from using append mode with full refreshes.

**Schema enforcement (constraints):** After the initial write, Delta `NOT NULL` constraints are applied to primary and foreign key columns (`id`, `show_id`), and `CHECK` constraints enforce that identifiers are positive integers. These persist as table metadata — any future write that violates them is rejected at the storage level, regardless of which notebook or pipeline performs the write.

In [0]:
import json

counts = {
    "bronze_shows":    spark.table(f"{CATALOG}.bronze.bronze_shows").count(),
    "bronze_episodes": spark.table(f"{CATALOG}.bronze.bronze_episodes").count(),
    "bronze_cast":     spark.table(f"{CATALOG}.bronze.bronze_cast").count(),
}

failed = [k for k, v in counts.items() if v == 0]
if failed:
    msg = f"Bronze validation FAILED — empty tables: {failed}"
    print(msg)
    dbutils.notebook.exit(json.dumps({"status": "FAILED", "reason": msg, "counts": counts}))

print("Bronze validation passed:", counts)
dbutils.notebook.exit(json.dumps({"status": "OK", "counts": counts}))